<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/golf/06_proportional_discrepancy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Golf putting 6 — Proportional discrepancy

Model each bin as having an additional probability of failure that scales the idealized angle-and-distance success probability downward. Keep Broadie’s distance tolerance and overshoot fixed for now.

## Setup

This notebook uses the PyMC / ArviZ packages provided by the current Colab environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

DATA_BASE = "https://raw.githubusercontent.com/opherdonchin/BayesShortCourse/main/golf/data"
golf = pd.read_csv(f"{DATA_BASE}/broadie_2018_putting.csv")
golf["rate"] = golf["made"] / golf["attempts"]

BALL_RADIUS_FT = (1.68 / 2) / 12
CUP_RADIUS_FT = (4.25 / 2) / 12

golf

In [ ]:
def plot_rate(draws, data, title, *, median_label="median", show_observed=True, ylim=(-0.05, 1.05), show_bounds=False):
    """Plot a probability/rate relationship against continuous putting distance."""
    x = data["distance_ft"].to_numpy()
    curve_dim = next(dim for dim in draws.dims if dim not in ("chain", "draw"))
    order = np.argsort(x)
    draws = draws.isel({curve_dim: order})
    x = x[order]

    median = draws.median(dim=("chain", "draw"))
    interval = draws.azstats.hdi(prob=0.90)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(
        x,
        interval.sel(ci_bound="lower"),
        interval.sel(ci_bound="upper"),
        alpha=0.22,
        label="90% HDI",
    )
    ax.plot(x, median, label=median_label)
    if show_observed:
        ax.scatter(x, data["rate"].to_numpy()[order], s=30, color="black", label="observed")
    if show_bounds:
        ax.axhline(0, color="0.6", linewidth=0.8, linestyle="--")
        ax.axhline(1, color="0.6", linewidth=0.8, linestyle="--")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        title=title,
    )
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.legend(fontsize=8, frameon=False)
    return ax

def plot_residuals(idata, data, var_name="p_base", title="Residuals"):
    """Plot observed minus posterior-median mechanistic probability."""
    fitted = idata["posterior"][var_name].median(dim=("chain", "draw")).to_numpy()
    x = data["distance_ft"].to_numpy()
    residual = data["rate"].to_numpy() - fitted
    order = np.argsort(x)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.plot(x[order], residual[order], marker="o")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Observed − fitted probability",
        title=title,
    )
    return ax

## Model

$$
p_j=p_{base,j}(1-\epsilon_j),\qquad 0\le \epsilon_j\le1.
$$

The $\epsilon_j$ values share a scale parameter, so the model can acknowledge local mismatch without allowing arbitrary upward corrections.

In [ ]:
coords = {"obs_id": golf["distance_ft"].to_numpy()}

with pm.Model(coords=coords) as model:
    distance = pm.Data("distance", golf["distance_ft"].to_numpy(), dims="obs_id")
    attempts = pm.Data("attempts", golf["attempts"].to_numpy(), dims="obs_id")\n
    sigma_angle_deg = pm.LogNormal("sigma_angle_deg", mu=np.log(2), sigma=0.7)
    sigma_distance = pm.LogNormal("sigma_distance", mu=np.log(0.10), sigma=0.7)
    distance_tolerance = 3.0
    overshot = 1.0

    sigma_angle_rad = sigma_angle_deg * np.pi / 180
    threshold_angle = pm.math.arcsin((CUP_RADIUS_FT - BALL_RADIUS_FT) / distance)
    p_angle = pm.Deterministic(
        "p_angle",
        2 * pm.math.invprobit(threshold_angle / sigma_angle_rad) - 1,
        dims="obs_id",
    )
    p_distance = pm.Deterministic(
        "p_distance",
        pm.math.invprobit(
            (distance_tolerance - overshot) / ((distance + overshot) * sigma_distance)
        )
        - pm.math.invprobit(
            -overshot / ((distance + overshot) * sigma_distance)
        ),
        dims="obs_id",
    )
    p_base = pm.Deterministic("p_base", p_angle * p_distance, dims="obs_id")
    sigma_epsilon = pm.HalfNormal("sigma_epsilon", sigma=0.02)
    epsilon = pm.Truncated(
        "epsilon",
        pm.Exponential.dist(lam=1 / sigma_epsilon),
        lower=0,
        upper=1,
        dims="obs_id",
    )
    p = pm.Deterministic("p", p_base * (1 - epsilon), dims="obs_id")

    made = pm.Binomial(
        "made",
        n=attempts,
        p=p,
        observed=golf["made"].to_numpy(),
        dims="obs_id",
    )

## Prior predictive check

Before fitting, ask what success rates the prior says are plausible across putting distance. The observed rates are shown only as a reference; they do not condition the prior.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=500,
        var_names=["made"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
plot_rate(
    prior["prior_predictive"]["made"] / golf["attempts"].to_numpy(),
    golf,
    "Prior predictive",
);

## Fit and diagnose

Do not interpret the scientific fit until the sampler diagnostics are acceptable.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED,
    )

print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))
azs.summary(
    idata,
    var_names=['sigma_angle_deg', 'sigma_distance', 'sigma_epsilon'],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

local_rhat = idata["posterior"]["epsilon"].azstats.rhat()
local_ess = idata["posterior"]["epsilon"].azstats.ess(method="bulk")
print(f"Worst epsilon R-hat: {float(local_rhat.max()):.3f}")
print(f"Smallest epsilon bulk ESS: {float(local_ess.min()):.0f}")

In [ ]:
azp.plot_trace_dist(idata, var_names=['sigma_angle_deg', 'sigma_distance', 'sigma_epsilon']);

## Posterior fit

This plot shows posterior uncertainty in the continuous underlying mechanism $p_{base}(x)$ **before** the local discrepancy terms are applied. That distinction matters: the full model can fit the data by using local corrections even when the shared mechanism still has systematic structure.

In [ ]:
plot_rate(
    idata["posterior"]["p_base"],
    golf,
    "Underlying mechanism before proportional discrepancy",
    median_label="posterior median",
);

## Posterior predictive check

Now generate new outcomes at the same putting distances from the **full fitted model**, including its local discrepancy terms. This uses the same graphical grammar as the prior predictive check; the difference is conditioning on the observed data.

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["made"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
plot_rate(
    idata["posterior_predictive"]["made"] / golf["attempts"].to_numpy(),
    golf,
    "Posterior predictive check",
);

## Model criticism

These residuals deliberately compare the observations with $p_{base}(x)$, before the bin-specific discrepancy terms. They ask whether the shared mechanism still misses a systematic feature that the local corrections are masking.

In [ ]:
plot_residuals(idata, golf, var_name="p_base", title="Residuals of the underlying mechanism");

## Decision: expand the geometry

The full model can reproduce the observations, but inspecting the **underlying mechanism before the local error terms** still suggests systematic discrepancy, particularly at short distances. Broadie’s three-foot distance-tolerance value was treated as known. The next scientifically interpretable revision is to estimate it instead of forcing the local error terms to compensate for it. **Next notebook:** learn distance tolerance.